In [2]:
"""
Purpose: Determined functions to ve included in the relevant python script through a general classification class.

Functions:
to open all json files that have the judicial process' ruling "activity", previously filtered. 
to filter the where the ruling is recorded within the json file and extract it.
to preprocess the text of the ruling, including unicode separation of chars and special chars such as accentuation. 
to replace colons, semicolons, apostrophes, with spaces. 
to retrieve the section in which the ruling is made and the context of the ruling using all caps text.
to classify the ruling in accordance to its type: accepted, denied, favorable.

Workflow:
1) Every json file containing the pre-filtered judicial process activity is opened and read
2) The ruling text is filtered. 
3) The ruling context "motivacion" and decision section are filtered using regex. 
4) The ruling decision section is matched to a type of decision
5) Two dicts are created for control. A general counter dict and a storing one in accordance of the ruling type
6) The recording dict is saved as a json file for further processing. 


Derived from: tc_rulings_classification.ipyn and exploration_tag_rulings.ipyn

"""

#import section
import json
import re
import glob
import os
from collections import defaultdict
import unicodedata

In [3]:
#Functions to open json files that include data of the judicial process including the text of the ruling
def open_rulings_json(file_path: str):
    """
        Function that opens a json file in accordance to the provided file_path
        Returns: file_name (judicial process number) and the data loaded from the json file.
    """
    #print(f"file path being processed with processed: {file_path}/n:")
    with open(file_path, 'rb') as file:
        if file.read(2) != b'[]':
            file_name = os.path.basename(file_path).strip('.json')
            #print(file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    json_data = json.load(f)
                    return file_name, json_data
                except json.JSONDecodeError:
                    print(f"Error: {file_name} is not a valid JSON file.")
        else:
            #print(f"file path empy, likely no ruling reached: {file_path}/n:")
            #print("file came empty so the case did not reach a ruling\n")
            #to avoid a crash of other function due to empty json files
            return "empty", "empty"

def retrieve_ruling(json_data) -> str:
    """
        Function that retrieves the ruling from the json data of a judial case, using the "actividad" tag
        Returns the text of the ruling of the relevant judicial process
    """
    for item in json_data:
        ruling = item.get("actividad")
    return ruling

In [106]:
#Normalization functions to strip accents, capitalize characters, eliminate commas and other puntuation. 
#Relevant for data preparation
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )

def norm_tipo(s: str) -> str:
    return re.sub(r"\s+", " ", strip_accents(s or "")).strip()

#new function that consolidates text caps, accent elimination, coma and other puntuation elimination and resulting space consolidation
def clean_text(text):
    # Uppercase + strip accents
    text_ed = norm_tipo(text)
    #Replace commas, semicolons, periods, and parentheses with spaces
    text_ed = re.sub(r'[:,.;()\-–—]', ' ', text_ed)
    #Collapse the resulting multiple spaces into a single space
    text_ed = re.sub(r'\s+', ' ', text_ed).upper()
    return text_ed.strip()
    

In [107]:
def slicing_rulings(ruling: str, number_of_chars: int):
    """
        Function that matches the section of the ruling in which the decision is made
        Returns the sliced section of the ruling where a decision is made to classify the type of ruling
        Returns the sliced section of the ruling (context of ruling) before the decision is made through match 
        Default: when no context is found, slice pursuant to a arbitrary number of chars. 
    """
    #Base Regex to identify the boilerplate phrase prior to which the decision is made
    RE_DECISION_BLOCK = re.compile(
        r"\bDECISION\b.{0,400}?\bADMINISTRANDO\s+JUSTICIA\s+(?:EN\s+)?NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+"
        r"DEL\s+ECUADOR\s+(?:Y\s+)?POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
        r"LAS\s+LEYES\s+DE\s+LA\s+REPUBLICA\s*(.*)", 
        re.DOTALL
    )

    # Alternative boilerplate when "DECISION" isn't explicitly written right before.
    RE_ADMINISTRANDO_BLOCK = re.compile(
        r"\bADMINISTRANDO\s+JUSTICIA\s+(?:EN\s+)?"
        r"NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+DEL\s+ECUADOR\s+(?:Y\s+)?"
        r"POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
        r"(?:DE\s+)?(?:LAS\s+)?LEYES\s+DE\s+LA\s+REPUBLICA\s*(.*)", 
        re.DOTALL
    )

    # Block to retrieve the ruling context.
    # Ordinal words (strong signal) -> allow up to ~14 filler words before the keyword.
    # Digits / cardinals (weak signal, e.g. "SEIS MESES", "ARTICULO 6") -> keyword must follow immediately.
    _KEY = r"(?:MOTIVACION|CONCLUSIONES\s+DEL\s+TRIBUNAL|CONSIDERACIONES\s+DEL\s+TRIBUNAL)"
    _LINK = r"(?:(?:DE\s+LA|DEL|DE)\s+)?"   # optional "DE LA" / "DEL" / "DE"
    
    RE_RULING_CONTEXT = re.compile(
        rf"""
        \b(?:QUINTO|QUINTA|SEXTO|SEXTA|SEPTIMO|SEPTIMA|OCTAVO|OCTAVA)\b          # ordinal words
            (?:\s+(?!\d)\w+){{0,14}}?          # up to 14 filler words (non-greedy, no digits)
            \s+(?={_KEY}\b)
        |
        (?<![\w])(?:[5678]|CINCO|SEIS|SIETE|OCHO)\s+ # digit / cardinal: keyword must follow right away
            (?={_LINK}{_KEY}\b)
        """,
        re.VERBOSE,
    )


    #Formating the ruling to match and classify its context. 
    clean_decision_upper = clean_text(ruling)
    #clean_decision_upper = clean_decision_ruling.upper()
    clean_decision_lower = clean_decision_upper.lower()

    #Determine whether we have a match and therefore identified the bolierplate phrase where the decision is made
    match_decision = RE_DECISION_BLOCK.search(clean_decision_upper)
    #Try to match where the ruling context begins withing the ruling
    match_context = RE_RULING_CONTEXT.search(clean_decision_upper)
    context_match = True if match_context else False

    if match_decision:
        #print("we have a match\n")
        #retrieve the part where the type of decision can be matched
        decision_text = match_decision.group(1)
        
        #retrieve the context before the decision is made for processing: 

        #index at which the decision section starts with boilerplate phrases
        decision_start = match_decision.start()
        #end of the ruling context where the decision section starts
        #end_ruling_context = decision_start
        #end_ruling_context = decision_start - (len(ruling) - len(clean_decision_ruling))
        
        if match_context:
            #if we found the decision context "motivación section of the tax court"
            context_start = match_context.start()
            #The ruling context we need for classification begins where the match starts
            #start_ruling_context = context_start
            #start_ruling_context = context_start - (len(ruling) - len(clean_decision_ruling))
            context_before_decision = clean_decision_lower[context_start:decision_start]
        else:
            #Couldnt fin the start of the ruling context for slicing purposes:
            #begining of the ruling context determined in accordance to a default length of chars
            #start is arbitrary
            start_ruling_context = max(0, decision_start - number_of_chars)
            #Slicing the ruling context for processing:
            context_before_decision = clean_decision_lower[start_ruling_context:decision_start]

        #Return the formated, lower-case decision text for further matching and the context before the decision
        return context_match, decision_text, context_before_decision

    else:
        #Else, try to match an alternative regex text, or fall withing a default section, where no match could be found for decision type 
        
        #Try to match in cases were the boilerplate phrase is not preceded by a DECISION section
        alt_decision_match = RE_ADMINISTRANDO_BLOCK.search(clean_decision_upper)

        if alt_decision_match:
            #retrieve the part where the type of decision can be matched
            decision_text = alt_decision_match.group(1)
            #formated_decision_text = clean_text(decision_text)
            
            #retrieve the context before the decision is made for processing: 
    
            #index at which the decision section starts with boilerplate phrases
            decision_start = alt_decision_match.start()
            #end of the context ruling just before the decision section starts
            #end_ruling_context = decision_start 
            #end_ruling_context = decision_start - (len(ruling) - len(clean_decision_ruling))
            
            if match_context:
                #Could find a context of match even for a case where no initial decision match was found
                context_start = match_context.start()
                #start_ruling_context = context_start - (len(clean_decision_ruling) - len(ruling))
                #start_ruling_context = context_start - (len(ruling) - len(clean_decision_ruling))
                context_before_decision = clean_decision_lower[context_start:decision_start]
            #end_ruling_context = decision_start - (len(ruling) - len(clean_decision_ruling))
            
            else:
                #Couldnt fin the start of the ruling context for slicing purposes:
                #begining of the ruling context determined in accordance to a desired length of chars
                #start is arbitrary
                start_ruling_context = max(0, decision_start - number_of_chars)
                #Slicing the ruling context for processing:
                context_before_decision = clean_decision_lower[start_ruling_context:decision_start]
                #slicing the resulting text for uniformity. The last part generally includes common to all phrases
                #thats why we slice the last part of the ruling context.
                #context_before_decision = context_before_decision[:number_of_chars]
    
            #Return the formated decision text for further matching and the context before the decision
            #print(formated_decision_text[:10], context_before_decision[:10])
            return context_match, decision_text, context_before_decision

        else:
            #No match could be found using both regex alternatives
            #print(f"we couldnt find the boilerplate phrase for {file_name}\n")
            #if no match happened, to avoid failures of the model we still need to return strings 
            #this will be classified as a no match in the dictionary for treacability and correction purposes
            fail_to_match = "failed to find and match the boilerplate decision text"
            failed_context= "no context before decision, no classification possible"
            return context_match, fail_to_match, failed_context

def matching_type_of_ruling_and_classify(general_dict: dict, ruling_classifier_dict: dict, file_name: str, formated_decision_text: str, context_before_decision:str):
    """
        Function that matches the decision section with a type of ruling (accept, denied, partial) or a default no match
        In accordance to the matching it classifies each ruling and also counts the number of each ruling type found
    """ 
    #Regex to match the three types of rulings from the taxpayer's perspective:
    #Accept the judicial challenge, deny the judicial challenge or partially accept the judicial challenge. 
    #general or most common patterns:
    RE_PARTIAL = re.compile(
        r"\bACEPT\w*\s+PARCIAL\b"
        r"|\bPARCIALMENTE\b"
        r"|\bACOGER?\s+PARCIAL\b"
    )
    
    RE_ACCEPT = re.compile(
        r"\b(?:ACEPT\w*|ACOG\w*)\s+(?:\w+\s+){0,5}DEMANDAS?\b"
        r"|\bSE\s+ACEPTA\b"
        r"|\bDECLARAR\s+(?:LA\s+)?VALIDEZ\b"
    )
    
    RE_DENY = re.compile(
        r"\b(?:NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*|INADM\w*)\s+(?:\w+\s+){0,5}DEMANDAS?\b"
        r"|\bSIN\s+LUGAR\b"
        r"|\bIMPROCEDENTE\b"
        r"|\bDECLARAR\s+SIN\s+LUGAR\b"
    )
    
    RE_ACCEPT_SPECIAL = re.compile(
        r"\bDECLARA\w*\s+(?:LA\s+)?NULIDAD\b"
        r"|\bDEJA\w*\s+SIN\s+EFECTO\s+(?:NI|Y|O)\s+VALIDEZ(?:\s+LEGAL)?\b"  # Fixed DEJAR? bug to DEJA\w*
    )
    
    RE_ACCEPT_COACTIVO = re.compile(
        r"\b(?:ACEPT\w*|ACOG\w*)\s+(?:LAS?\s+)?EXCEPCION(?:ES)?\s+(?:\w+\s+){0,7}PROCE\w*\s+COACTIVO[S]?\b"
    )
    
    RE_DENY_COACTIVO = re.compile(
        r"\b(?:NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(?:LAS?\s+)?EXCEPCION(?:ES)?\s+(?:\w+\s+){0,7}PROCE\w*\s+COACTIVO[S]?\b"
        r"|\b(?:NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(?:LAS?\s+)?EXCEPCION(?:ES)?\b"
    )


    #matching section
    if RE_PARTIAL.search(formated_decision_text):
        general_dict["partial"] += 1
        #print(f"partial {d}")
        #once confirmed the ruling type:
        ruling_classifier_dict["partial"][file_name] = context_before_decision

    elif RE_ACCEPT.search(formated_decision_text):
        general_dict["accept"] += 1
        #print(f"accept {d}")
        #once confirmed extract a section of the tax ruling:
        ruling_classifier_dict["accept"][file_name] = context_before_decision

    elif RE_DENY.search(formated_decision_text):
        general_dict["denied"] += 1
        #print(f"denied {d}")
        #once confirmed extract a section of the tax ruling:
        ruling_classifier_dict["denied"][file_name] = context_before_decision

    else:
        #matching special cases: 
        if  RE_ACCEPT_COACTIVO.search(formated_decision_text):
            general_dict["accept"] += 1
            #print(f"accept {d}")
            #once confirmed extract a section of the tax ruling:
            ruling_classifier_dict["accept"][file_name] = context_before_decision
        
        elif RE_DENY_COACTIVO.search(formated_decision_text):
            general_dict["denied"] += 1
            #print(f"denied {d}")
            #once confirmed extract a section of the tax ruling:
            ruling_classifier_dict["denied"][file_name] = context_before_decision

        elif  RE_ACCEPT_SPECIAL.search(formated_decision_text):
            general_dict["accept"] += 1
            #print(f"accept {d}")
            #once confirmed extract a section of the tax ruling:
            ruling_classifier_dict["accept"][file_name] = context_before_decision

        else:
            print(f"we couldnt match the decision type for {file_name}\n")
            print(f"decision_text: {formated_decision_text}")
            general_dict["no matches found"] += 1
            ruling_classifier_dict["no matches found"][file_name] = context_before_decision


    #return the dictionaries:
    return general_dict, ruling_classifier_dict

    

In [108]:
"""Processing and taggin and classifying routine"""

#path to filtered tax ruling json files (path leverages on volume mounting)
path_to_files = "/home/jovyan/data/"

# retrieval of json files
json_files = glob.glob(os.path.join(path_to_files, "*.json"))

In [109]:
# creating dictionaries for counting ruling types and classifying rulings accordingly
general_dict = {"accept": 0, "partial": 0, "denied": 0, "no matches found": 0}
context_match = defaultdict(dict)
ruling_classifier_dict = defaultdict(dict)
#specify the desired number of chars for the ruling_context
number_of_chars = 12000
#empty ruling counter:
empty_ruling=0

In [110]:
#logic implementation: 
for file_path in json_files:
    file_name, json_data = open_rulings_json(file_path)
    if file_name != "empty":
        ruling = retrieve_ruling(json_data)
        match_context, formated_decision_text, context_before_decision= slicing_rulings(ruling, number_of_chars)
        if match_context == False:
            context_match[file_name] = context_before_decision
        general_dict, ruling_classifier_dict =  matching_type_of_ruling_and_classify(general_dict, ruling_classifier_dict, file_name, formated_decision_text, context_before_decision)
    else:
        empty_ruling+=1
    

we couldnt match the decision type for 17510202100113

decision_text: ACEPTA EL RECURSO DE QUEJA PRESENTADO POR EL AB DANIEL HERNANDEZ VERA PROCURADOR JUDICIAL DEL SENOR JUAN MANUEL DURINI REPRESENTANTE LEGAL DE LA COMPANIA ENCHAPES DECORATIVOS S A ENDESA Y EN CONSECUENCIA IMPONE UNA SANCION PECUNIARIA DE USD 150 00 CIENTO CINCUENTA DOLARES DE LOS ESTADOS UNIDOS DE NORTEAMERICA A LA FUNCIONARIA RECAUDADORA DEL GOBIERNO AUTONOMO DESCENTRALIZADO MUNICIPAL DE ELOY ALFARO QUE ESTUVO EN TALES FUNCIONES DESDE EL 11 DE ENERO DEL 2019 HASTA EL 18 DE ENERO DEL 2019 FECHA EN QUE SE CUMPLIA EL PLAZO DE 5 DIAS PARA LA REMISION DE LAS EXCEPCIONES POR LAS OMISIONES SENALADAS EN ESTA SENTENCIA EN EJERCICIO DE SUS FUNCIONES COMO FUNCIONARIA PUBLICA EN CALIDAD DE RESPONSABLE DEL INCUMPLIMIENTO SENALADO EN EL ARTICULO 279 DEL CODIGO TRIBUTARIO SE DISPONE QUE EN EL TERMINO DE 8 DIAS LA FUNCIONARIA EJECUTORA ACTUAL O EN SU DEFECTO LA MAXIMA AUTORIDAD DEL MUNICIPIO REMITA DICHAS EXCEPCIONES CON LOS DOCUMEN

In [111]:
print(general_dict)
distinct_counts = {key: len(set(values)) for key, values in ruling_classifier_dict.items()}
print(f"likely no ruling issued: {empty_ruling}")

{'accept': 410, 'partial': 156, 'denied': 230, 'no matches found': 11}
likely no ruling issued: 374


In [112]:
assert general_dict["accept"] == distinct_counts["accept"]

In [113]:
assert general_dict["partial"] == distinct_counts["partial"]

In [114]:
assert general_dict["denied"] == distinct_counts["denied"]

In [115]:
assert general_dict["no matches found"] == distinct_counts["no matches found"]

In [116]:
print(ruling_classifier_dict["partial"]["17510201900371"][:200])

6 motivacion del fallo 6 1 de la congruencia en el escenario del estado constitucional de derechos y justicia resalta aquel derecho contenido en el articulo 173 de la constitucion de la republica el c


In [118]:
print(ruling_classifier_dict["partial"]["17510201900371"][-220:])

r lo dicho se ratifica la glosa por lo que no se reconocen las deducciones por el valor de usd 505 647 83 consignadas por la actora en la casilla 7203 de su declaracion de impuesto a la renta del ejercicio fiscal 2016 7 


In [119]:
print(len(ruling_classifier_dict["partial"]["17510201900371"]))

147527


In [47]:
#After the data extraction and wrangling step is done, we have to export the resulting json file: 
output_dir = "/home/jovyan/prepared_extracted/"
output_file = os.path.join(output_dir, "extracted_classified_tr.json")
with open(output_file, "w", encoding='utf8') as json_file:
    json.dump(ruling_classifier_dict, json_file, indent=4, ensure_ascii=False)

In [120]:
print(len(context_match))

160


In [121]:
for key in context_match:
    print(key)

17510201900283
17510202000138
17510201900368
17510201900190
17510201900169
17510202000143
17510202100201
17510201900364
17510202000088
17510201900333
17510202100057
17510201900173
17510202000362
17510202000227
17510202000118
17510202000374
17510202100105
17510202100001
17510202100144
17510201900172
17510201900349
17510202000334
17510202100056
17510202000271
17510201900332
17510202100304
17510202100083
17510202100200
17510202000162
17510202000135
17510201900216
17510201900304
17510201900257
17510202000154
17510202000338
17510201900152
17510202000314
17510201900390
17510201900144
17510202100172
17510201900289
17510201900323
17510202100084
17510201900374
17510202100354
17510202000221
17510202000372
17510201900358
17510202000260
17510202100319
17510202100122
17510201900397
17510201900381
17510202100026
17510201900339
17510202000128
17510202100118
17510202100323
17510202000112
17510201900211
17510202000145
17510202000113
17510202100158
17510202000390
17510202000386
17510202000017
1751020190